# Holistic Data Preparer (Final Project)

Customer credit risk preprocessing project.

In [65]:
import json
import sqlite3

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.preprocessing import (
    Binarizer,
    FunctionTransformer,
    LabelEncoder,
    MaxAbsScaler,
    MinMaxScaler,
    Normalizer,
    OneHotEncoder,
    OrdinalEncoder,
    PowerTransformer,
    RobustScaler,
    StandardScaler,
)
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)

## Part A: Conceptual Foundation

### 1. Short Notes

**What is Data Analysis?**  
Data analysis means studying data to understand it and get useful information.

**How to Plan a Data Science Project**  
1. Understand the problem.  
2. Collect the data.  
3. Clean the data.  
4. Explore the data.  
5. Create features.  
6. Train and test the model.  
7. Explain the result.

**How to Frame a Machine Learning Problem**  
Here `default_flag` is the target, so this is a supervised binary classification problem.

### 2. Tensors with NumPy

A tensor is a multi-dimensional array.

- 0D tensor: single value
- 1D tensor: vector
- 2D tensor: matrix
- 3D tensor: group of matrices

In [66]:
scalar = np.array(10)
vector = np.array([2, 4, 6, 8])
matrix = np.array([[1, 2], [3, 4], [5, 6]])
tensor_3d = np.array([
    [[1, 2], [3, 4]],
    [[5, 6], [7, 8]],
])

print("0D tensor:", scalar)
print("Shape:", scalar.shape, "Dimensions:", scalar.ndim)
print()

print("1D tensor:")
print(vector)
print("Shape:", vector.shape, "Dimensions:", vector.ndim)
print()

print("2D tensor:")
print(matrix)
print("Shape:", matrix.shape, "Dimensions:", matrix.ndim)
print()

print("3D tensor:")
print(tensor_3d)
print("Shape:", tensor_3d.shape, "Dimensions:", tensor_3d.ndim)

0D tensor: 10
Shape: () Dimensions: 0

1D tensor:
[2 4 6 8]
Shape: (4,) Dimensions: 1

2D tensor:
[[1 2]
 [3 4]
 [5 6]]
Shape: (3, 2) Dimensions: 2

3D tensor:
[[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]]
Shape: (2, 2, 2) Dimensions: 3


Interpretation: This shows examples of 0D, 1D, 2D, and 3D arrays in NumPy.

## Part B: Data Acquisition

In [67]:
transactions_df = pd.read_csv("data/customer_transactions.csv")
transactions_df.head()

,customer_id,age,annual_income,loan_amount,loan_purpose,credit_score,transaction_count,spending_ratio,default_flag
0,CUST001,24.0,325717.20,190048.17,Home,625.0,105,62.89,0
1,CUST002,50.0,336131.76,295795.09,Business,707.0,49,31.40,0
2,CUST003,45.0,179537.36,111123.85,Other,660.0,235,45.96,0
3,CUST004,37.0,413197.23,170131.20,Business,667.0,166,53.10,1
4,CUST005,37.0,283165.26,201232.36,Education,696.0,209,47.18,1


Interpretation: This CSV file is the main dataset.

In [68]:
with open("data/customer_metadata.json", "r", encoding="utf-8") as file:
    metadata = json.load(file)

metadata_df = pd.DataFrame(metadata)
metadata_df.head()

,customer_id,gender,region,education_level,employment_type,join_date
0,CUST001,Female,West,Graduate,Unemployed,2023-03-31
1,CUST002,Other,North,Graduate,Unemployed,2019-09-22
2,CUST003,None,South,Primary,Salaried,2024-05-17
3,CUST004,Male,East,Graduate,Salaried,2022-08-31
4,CUST005,Male,South,Primary,Self-Employed,2019-03-16


Interpretation: This JSON file has customer details.

In [69]:
conn = sqlite3.connect("data/loan_repayment.db")
repayment_df = pd.read_sql_query("SELECT * FROM repayment_history", conn)
conn.close()

repayment_df.head()

,customer_id,repayment_history
0,CUST001,1
1,CUST002,2
2,CUST003,2
3,CUST004,4
4,CUST005,9


Interpretation: This SQL table has repayment history.

In [70]:
def fetch_dummy_api_data(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        response = json.load(file)
    return pd.DataFrame(response["data"])

api_df = fetch_dummy_api_data("data/api_economic_indicators.json")
api_df

,region,regional_risk_index,local_unemployment_rate
0,North,48,4.8
1,South,61,6.1
2,East,55,5.4
3,West,43,4.2


Interpretation: This file acts like external API data.

In [71]:
df = transactions_df.merge(metadata_df, on="customer_id", how="left")
df = df.merge(repayment_df, on="customer_id", how="left")
df = df.merge(api_df, on="region", how="left")

print("Merged shape:", df.shape)
df.head()

Merged shape: (60, 17)


,customer_id,age,annual_income,loan_amount,loan_purpose,credit_score,transaction_count,spending_ratio,default_flag,gender,region,education_level,employment_type,join_date,repayment_history,regional_risk_index,local_unemployment_rate
0,CUST001,24.0,325717.20,190048.17,Home,625.0,105,62.89,0,Female,West,Graduate,Unemployed,2023-03-31,1,43,4.2
1,CUST002,50.0,336131.76,295795.09,Business,707.0,49,31.40,0,Other,North,Graduate,Unemployed,2019-09-22,2,48,4.8
2,CUST003,45.0,179537.36,111123.85,Other,660.0,235,45.96,0,None,South,Primary,Salaried,2024-05-17,2,61,6.1
3,CUST004,37.0,413197.23,170131.20,Business,667.0,166,53.10,1,Male,East,Graduate,Salaried,2022-08-31,4,55,5.4
4,CUST005,37.0,283165.26,201232.36,Education,696.0,209,47.18,1,Male,South,Primary,Self-Employed,2019-03-16,9,61,6.1


Interpretation: Now all source data is combined in one dataframe.

## Part C: Data Understanding and Cleaning

In [72]:
df.info()
display(df.describe(include="all").T)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customer_id              60 non-null     object 
 1   age                      56 non-null     float64
 2   annual_income            57 non-null     float64
 3   loan_amount              58 non-null     float64
 4   loan_purpose             60 non-null     object 
 5   credit_score             57 non-null     float64
 6   transaction_count        60 non-null     int64  
 7   spending_ratio           60 non-null     float64
 8   default_flag             60 non-null     int64  
 9   gender                   57 non-null     object 
 10  region                   60 non-null     object 
 11  education_level          60 non-null     object 
 12  employment_type          57 non-null     object 
 13  join_date                60 non-null     object 
 14  repayment_history        60 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,60,60,CUST001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,56.0,NaN,NaN,NaN,39.607143,10.351423,22.0,30.5,40.0,49.0,58.0
annual_income,57.0,NaN,NaN,NaN,541356.788947,396919.426333,150542.91,326748.36,424700.56,535446.65,2400000.0
loan_amount,58.0,NaN,NaN,NaN,343032.042069,241097.202778,54973.03,174989.6675,274312.965,404543.27,1200000.0
loan_purpose,60,5,Home,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credit_score,57.0,NaN,NaN,NaN,682.964912,81.46712,345.0,650.0,685.0,723.0,846.0
transaction_count,60.0,NaN,NaN,NaN,120.783333,59.884661,21.0,71.25,128.0,168.0,260.0
spending_ratio,60.0,NaN,NaN,NaN,41.8405,17.672111,15.62,30.3825,36.145,53.17,92.5
default_flag,60.0,NaN,NaN,NaN,0.2,0.403376,0.0,0.0,0.0,0.0,1.0
gender,57,3,Male,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Interpretation: The dataset has both numeric and categorical columns.

In [73]:
try:
    from ydata_profiling import ProfileReport

    profile = ProfileReport(df, title="Customer Credit Risk Data Quality Report", minimal=True)
    profile.to_file("outputs/data_quality_report.html")
    print("Profiling report saved in outputs/data_quality_report.html")
except Exception as error:
    print("Profiling report could not be created:", error)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 17/17 [00:00<00:00, 960.89it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profiling report saved in outputs/data_quality_report.html


Interpretation: A profiling report was created and saved in the outputs folder.

In [74]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0]

age                4
employment_type    3
annual_income      3
credit_score       3
gender             3
loan_amount        2
dtype: int64

Interpretation: Missing values are present in the required columns.

In [75]:
simple_num_df = df.copy()
num_cols = ["age", "annual_income", "loan_amount", "credit_score"]

num_imputer = SimpleImputer(strategy="median")
simple_num_df[num_cols] = num_imputer.fit_transform(simple_num_df[num_cols])

simple_num_df[num_cols].isna().sum()

age              0
annual_income    0
loan_amount      0
credit_score     0
dtype: int64

Interpretation: Median imputation filled missing values in numeric columns.

In [76]:
simple_cat_df = df.copy()
cat_cols = ["employment_type"]

cat_imputer = SimpleImputer(strategy="most_frequent")
simple_cat_df[cat_cols] = cat_imputer.fit_transform(simple_cat_df[cat_cols])

simple_cat_df[cat_cols].isna().sum()

employment_type    3
dtype: int64

Interpretation: Mode imputation filled missing values in categorical columns.

In [77]:
gender_df = df.copy()
gender_df[["gender"]] = SimpleImputer(strategy="most_frequent").fit_transform(gender_df[["gender"]])

gender_df["gender"].value_counts(dropna=False)

gender
Male      31
Female    25
None       3
Other      1
Name: count, dtype: int64

Interpretation: Gender missing values were filled with the most frequent value.

In [78]:
sample_df = df.copy()
sample_df["annual_income_missing_flag"] = sample_df["annual_income"].isna().astype(int)

observed_values = sample_df["annual_income"].dropna().to_numpy()
random_state = np.random.default_rng(7)
sample_df.loc[sample_df["annual_income"].isna(), "annual_income"] = random_state.choice(
    observed_values,
    size=sample_df["annual_income"].isna().sum(),
    replace=True,
)

sample_df[["annual_income", "annual_income_missing_flag"]].head(10)

,annual_income,annual_income_missing_flag
0,325717.20,0
1,336131.76,0
2,179537.36,0
3,413197.23,0
4,283165.26,0
5,354206.53,0
6,1850000.00,0
7,425240.31,0
8,211452.16,0
9,397984.96,1


Interpretation: Random sample imputation fills missing values using existing values from the same column.

In [79]:
knn_df = df.copy()
knn_cols = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "repayment_history",
    "transaction_count",
    "spending_ratio",
    "regional_risk_index",
    "local_unemployment_rate",
]

knn_imputer = KNNImputer(n_neighbors=5)
knn_df[knn_cols] = knn_imputer.fit_transform(knn_df[knn_cols])

knn_df[knn_cols].isna().sum()

age                        0
annual_income              0
loan_amount                0
credit_score               0
repayment_history          0
transaction_count          0
spending_ratio             0
regional_risk_index        0
local_unemployment_rate    0
dtype: int64

Interpretation: KNN imputation fills missing values based on similar rows.

In [80]:
mice_df = df.copy()
mice_df[["gender", "employment_type"]] = SimpleImputer(strategy="most_frequent").fit_transform(
    mice_df[["gender", "employment_type"]]
)

mice_cols = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "repayment_history",
    "transaction_count",
    "spending_ratio",
]

mice_imputer = IterativeImputer(random_state=42, max_iter=15)
mice_df[mice_cols] = mice_imputer.fit_transform(mice_df[mice_cols])

mice_df[mice_cols].isna().sum()

age                  0
annual_income        0
loan_amount          0
credit_score         0
repayment_history    0
transaction_count    0
spending_ratio       0
dtype: int64

Interpretation: MICE imputation estimates missing values using other columns.

In [81]:
complete_case_df = df.dropna().copy()

print("Original rows:", len(df))
print("Rows after complete case analysis:", len(complete_case_df))
print("Rows dropped:", len(df) - len(complete_case_df))

Original rows: 60
Rows after complete case analysis: 42
Rows dropped: 18


Interpretation: Complete case analysis drops rows with missing values.

In [82]:
clean_df = df.copy()
clean_df["annual_income_missing_flag"] = clean_df["annual_income"].isna().astype(int)
clean_df["credit_score_missing_flag"] = clean_df["credit_score"].isna().astype(int)

clean_df[["gender", "employment_type"]] = SimpleImputer(strategy="most_frequent").fit_transform(
    clean_df[["gender", "employment_type"]]
)

final_knn_cols = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "repayment_history",
    "transaction_count",
    "spending_ratio",
    "regional_risk_index",
    "local_unemployment_rate",
]

clean_df[final_knn_cols] = KNNImputer(n_neighbors=5).fit_transform(clean_df[final_knn_cols])

clean_df.isna().sum().sort_values(ascending=False).head(10)

gender               3
employment_type      3
customer_id          0
annual_income        0
age                  0
credit_score         0
loan_amount          0
transaction_count    0
spending_ratio       0
default_flag         0
dtype: int64

Interpretation: This cleaned dataframe will be used in the next steps.

## Part D: Outlier Handling

In [83]:
outlier_cols = ["annual_income", "loan_amount", "credit_score", "spending_ratio"]
z_scores = ((clean_df[outlier_cols] - clean_df[outlier_cols].mean()) / clean_df[outlier_cols].std()).abs()
z_score_outliers = (z_scores > 3).sum()
z_score_outliers

annual_income     2
loan_amount       1
credit_score      1
spending_ratio    0
dtype: int64

Interpretation: Z-score helps find extreme values.

In [84]:
iqr_counts = {}

for col in outlier_cols:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    iqr_counts[col] = ((clean_df[col] < lower) | (clean_df[col] > upper)).sum()

pd.Series(iqr_counts)

annual_income     5
loan_amount       4
credit_score      5
spending_ratio    2
dtype: int64

Interpretation: IQR also finds outliers and works well for skewed data.

In [85]:
percentile_df = clean_df.copy()
winsor_df = clean_df.copy()

for col in outlier_cols:
    p1 = percentile_df[col].quantile(0.01)
    p99 = percentile_df[col].quantile(0.99)
    percentile_df[col] = percentile_df[col].clip(p1, p99)

    p5 = winsor_df[col].quantile(0.05)
    p95 = winsor_df[col].quantile(0.95)
    winsor_df[col] = winsor_df[col].clip(p5, p95)

pd.DataFrame(
    {
        "original_max": clean_df[outlier_cols].max(),
        "percentile_max": percentile_df[outlier_cols].max(),
        "winsorized_max": winsor_df[outlier_cols].max(),
    }
)

,original_max,percentile_max,winsorized_max
annual_income,2400000.0,2075500.000,1.102797e+06
loan_amount,1200000.0,1103570.459,9.322529e+05
credit_score,846.0,839.510,7.884500e+02
spending_ratio,92.5,89.963,7.173700e+01


Interpretation: Capping and winsorization both reduce extreme values.

In [86]:
outlier_handled_df = clean_df.copy()
winsor_limits = {}

for col in outlier_cols:
    lower = outlier_handled_df[col].quantile(0.05)
    upper = outlier_handled_df[col].quantile(0.95)
    winsor_limits[col] = (round(lower, 2), round(upper, 2))
    outlier_handled_df[col] = outlier_handled_df[col].clip(lower, upper)

winsor_limits

{'annual_income': (np.float64(212564.05), np.float64(1102797.46)),
 'loan_amount': (np.float64(110720.47), np.float64(932252.93)),
 'credit_score': (np.float64(570.55), np.float64(788.45)),
 'spending_ratio': (np.float64(19.8), np.float64(71.74))}

Interpretation: I used winsorization in the cleaned dataset.

## Part E: Feature Engineering

In [87]:
feature_df = outlier_handled_df.copy()
feature_df["join_date"] = pd.to_datetime(feature_df["join_date"])

feature_df["join_year"] = feature_df["join_date"].dt.year
feature_df["join_month"] = feature_df["join_date"].dt.month
feature_df["join_day"] = feature_df["join_date"].dt.day
feature_df["join_weekday"] = feature_df["join_date"].dt.day_name()

feature_df[["customer_id", "join_date", "join_year", "join_month", "join_day", "join_weekday"]].head()

,customer_id,join_date,join_year,join_month,join_day,join_weekday
0,CUST001,2023-03-31,2023,3,31,Friday
1,CUST002,2019-09-22,2019,9,22,Sunday
2,CUST003,2024-05-17,2024,5,17,Friday
3,CUST004,2022-08-31,2022,8,31,Wednesday
4,CUST005,2019-03-16,2019,3,16,Saturday


Interpretation: Join date was split into year, month, day, and weekday.

In [88]:
ordinal_encoder = OrdinalEncoder(categories=[["Primary", "Secondary", "Graduate", "Post-Graduate"]])
feature_df["education_level_encoded"] = ordinal_encoder.fit_transform(feature_df[["education_level"]]).astype(int)

gender_encoder = LabelEncoder()
feature_df["gender_encoded"] = gender_encoder.fit_transform(feature_df["gender"])

one_hot_cols = ["region", "loan_purpose"]
one_hot_df = pd.get_dummies(feature_df[one_hot_cols], prefix=one_hot_cols, dtype=int)
feature_df = pd.concat([feature_df, one_hot_df], axis=1)

feature_df.head()

,customer_id,age,annual_income,loan_amount,loan_purpose,credit_score,transaction_count,spending_ratio,default_flag,gender,region,education_level,employment_type,join_date,repayment_history,regional_risk_index,local_unemployment_rate,annual_income_missing_flag,credit_score_missing_flag,join_year,join_month,join_day,join_weekday,education_level_encoded,gender_encoded,region_East,region_North,region_South,region_West,loan_purpose_Business,loan_purpose_Car,loan_purpose_Education,loan_purpose_Home,loan_purpose_Other
0,CUST001,24.0,325717.2000,190048.17,Home,625.0,105.0,62.89,0,Female,West,Graduate,Unemployed,2023-03-31,1.0,43.0,4.2,0,0,2023,3,31,Friday,2,0,0,0,0,1,0,0,0,1,0
1,CUST002,50.0,336131.7600,295795.09,Business,707.0,49.0,31.40,0,Other,North,Graduate,Unemployed,2019-09-22,2.0,48.0,4.8,0,0,2019,9,22,Sunday,2,2,0,1,0,0,1,0,0,0,0
2,CUST003,45.0,212564.0495,111123.85,Other,660.0,235.0,45.96,0,None,South,Primary,Salaried,2024-05-17,2.0,61.0,6.1,0,0,2024,5,17,Friday,0,3,0,0,1,0,0,0,0,0,1
3,CUST004,37.0,413197.2300,170131.20,Business,667.0,166.0,53.10,1,Male,East,Graduate,Salaried,2022-08-31,4.0,55.0,5.4,0,0,2022,8,31,Wednesday,2,1,1,0,0,0,1,0,0,0,0
4,CUST005,37.0,283165.2600,201232.36,Education,696.0,209.0,47.18,1,Male,South,Primary,Self-Employed,2019-03-16,9.0,61.0,6.1,0,0,2019,3,16,Saturday,0,1,0,0,1,0,0,0,1,0,0


Interpretation: Encoding was done based on the column type.

In [89]:
feature_df["income_group"] = pd.cut(
    feature_df["annual_income"],
    bins=4,
    labels=["Low", "Lower-Mid", "Upper-Mid", "High"],
)

feature_df["credit_score_flag"] = Binarizer(threshold=700).fit_transform(
    feature_df[["credit_score"]]
).ravel().astype(int)

feature_df["repayment_history_quantile_bin"] = pd.qcut(
    feature_df["repayment_history"],
    q=4,
    labels=False,
    duplicates="drop",
)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
feature_df["transaction_cluster_raw"] = kmeans.fit_predict(feature_df[["transaction_count"]])

cluster_order = (
    feature_df.groupby("transaction_cluster_raw")["transaction_count"]
    .mean()
    .sort_values()
    .index
    .tolist()
)
cluster_map = {cluster: idx for idx, cluster in enumerate(cluster_order)}
feature_df["transaction_kmeans_bin"] = feature_df["transaction_cluster_raw"].map(cluster_map)

feature_df[
    [
        "annual_income",
        "income_group",
        "credit_score",
        "credit_score_flag",
        "repayment_history",
        "repayment_history_quantile_bin",
        "transaction_count",
        "transaction_kmeans_bin",
    ]
].head()

c:\Users\patel\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,annual_income,income_group,credit_score,credit_score_flag,repayment_history,repayment_history_quantile_bin,transaction_count,transaction_kmeans_bin
0,325717.2000,Low,625.0,0,1.0,0,105.0,1
1,336131.7600,Low,707.0,1,2.0,1,49.0,0
2,212564.0495,Low,660.0,0,2.0,1,235.0,2
3,413197.2300,Low,667.0,0,4.0,2,166.0,1
4,283165.2600,Low,696.0,0,9.0,3,209.0,2


Interpretation: Binning and binarization changed continuous values into groups.

## Part F: Feature Scaling

In [90]:
scale_cols = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "repayment_history",
    "transaction_count",
    "spending_ratio",
    "regional_risk_index",
    "local_unemployment_rate",
]

scalers = {
    "Standardization": StandardScaler(),
    "Normalization": Normalizer(),
    "MinMax Scaling": MinMaxScaler(),
    "MaxAbs Scaling": MaxAbsScaler(),
    "Robust Scaling": RobustScaler(),
}

for name, scaler in scalers.items():
    scaled_values = scaler.fit_transform(feature_df[scale_cols])
    preview_df = pd.DataFrame(scaled_values, columns=scale_cols)
    print(name)
    display(preview_df.head(3))
    print("-" * 60)

Standardization


,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,regional_risk_index,local_unemployment_rate
0,-1.569598,-0.742064,-0.680444,-1.045400,-0.892202,-0.265786,1.379444,-1.506407,-1.528720
1,1.050878,-0.699356,-0.185167,0.341027,-0.430718,-1.208809,-0.627124,-0.796953,-0.708542
2,0.546941,-1.206087,-1.050095,-0.453632,-0.430718,1.923373,0.300651,1.047627,1.068509


------------------------------------------------------------
Normalization


,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,regional_risk_index,local_unemployment_rate
0,0.000064,0.863724,0.503962,0.001657,0.000003,0.000278,0.000167,0.000114,0.000011
1,0.000112,0.750714,0.660626,0.001579,0.000004,0.000109,0.000070,0.000107,0.000011
2,0.000188,0.886203,0.463288,0.002752,0.000008,0.000980,0.000192,0.000254,0.000025


------------------------------------------------------------
MinMax Scaling


...

------------------------------------------------------------
MaxAbs Scaling


,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,regional_risk_index,local_unemployment_rate
0,0.413793,0.295355,0.203859,0.792695,0.090909,0.403846,0.876675,0.704918,0.688525
1,0.862069,0.304799,0.317291,0.896696,0.181818,0.188462,0.437710,0.786885,0.786885
2,0.775862,0.192750,0.119199,0.837085,0.181818,0.903846,0.640674,1.000000,1.000000


------------------------------------------------------------
Robust Scaling


,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,regional_risk_index,local_unemployment_rate
0,-1.060000,-0.390621,-0.376337,-0.930035,-0.666667,-0.237726,1.173670,-0.923077,-0.923077
1,0.673333,-0.352994,0.095942,0.228975,-0.333333,-0.816537,-0.208228,-0.538462,-0.461538
2,0.340000,-0.799435,-0.728824,-0.435336,-0.333333,1.105943,0.430719,0.461538,0.538462


------------------------------------------------------------


Interpretation: This shows how each scaler changes the numeric data.

## Part G: Feature Construction and Transformation

In [91]:
final_feature_df = feature_df.copy()
final_feature_df["debt_to_income_ratio"] = final_feature_df["loan_amount"] / final_feature_df["annual_income"].replace(0, np.nan)
final_feature_df["average_monthly_transactions"] = final_feature_df["transaction_count"] / 6
final_feature_df["spending_to_income_ratio"] = final_feature_df["spending_ratio"] / 100

final_feature_df[["debt_to_income_ratio", "average_monthly_transactions", "spending_to_income_ratio"]].head()

,debt_to_income_ratio,average_monthly_transactions,spending_to_income_ratio
0,0.583476,17.500000,0.6289
1,0.879997,8.166667,0.3140
2,0.522778,39.166667,0.4596
3,0.411743,27.666667,0.5310
4,0.710653,34.833333,0.4718


Interpretation: These new features add extra useful information.

In [92]:
log_transformer = FunctionTransformer(np.log1p, validate=False)
reciprocal_transformer = FunctionTransformer(lambda x: 1 / (x + 1), validate=False)
sqrt_transformer = FunctionTransformer(np.sqrt, validate=False)

final_feature_df["spending_log"] = log_transformer.transform(final_feature_df["spending_ratio"])
final_feature_df["spending_reciprocal"] = reciprocal_transformer.transform(final_feature_df["spending_ratio"])
final_feature_df["spending_sqrt"] = sqrt_transformer.transform(final_feature_df["spending_ratio"])

final_feature_df[["spending_ratio", "spending_log", "spending_reciprocal", "spending_sqrt"]].head()

,spending_ratio,spending_log,spending_reciprocal,spending_sqrt
0,62.89,4.157163,0.015652,7.930322
1,31.40,3.478158,0.030864,5.603570
2,45.96,3.849296,0.021295,6.779381
3,53.10,3.990834,0.018484,7.286975
4,47.18,3.874944,0.020756,6.868770


Interpretation: These transformations help reduce skewness.

In [93]:
boxcox = PowerTransformer(method="box-cox")
yeojohnson = PowerTransformer(method="yeo-johnson")

final_feature_df["annual_income_boxcox"] = boxcox.fit_transform(final_feature_df[["annual_income"]])
final_feature_df["loan_amount_yeojohnson"] = yeojohnson.fit_transform(final_feature_df[["loan_amount"]])

final_feature_df[["annual_income", "annual_income_boxcox", "loan_amount", "loan_amount_yeojohnson"]].head()

,annual_income,annual_income_boxcox,loan_amount,loan_amount_yeojohnson
0,325717.2000,-0.742288,190048.17,-0.642590
1,336131.7600,-0.663278,295795.09,0.146948
2,212564.0495,-1.893803,111123.85,-1.702387
3,413197.2300,-0.163859,170131.20,-0.851717
4,283165.2600,-1.103277,201232.36,-0.536430


Interpretation: Power transformations make the data more balanced.

In [94]:
numeric_features = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "repayment_history",
    "transaction_count",
    "spending_ratio",
]
categorical_features = ["employment_type", "region", "loan_purpose"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

prepared_array = preprocessor.fit_transform(df)
print("ColumnTransformer output shape:", prepared_array.shape)

ColumnTransformer output shape: (60, 20)


Interpretation: ColumnTransformer handles numeric and categorical columns together.

## Part H: Final Deliverable

In [95]:
final_dataset = final_feature_df.copy()

final_dataset["join_weekday_encoded"] = LabelEncoder().fit_transform(final_dataset["join_weekday"])
final_dataset["income_group_encoded"] = final_dataset["income_group"].map(
    {"Low": 0, "Lower-Mid": 1, "Upper-Mid": 2, "High": 3}
).astype(int)

employment_dummies = pd.get_dummies(final_dataset["employment_type"], prefix="employment_type", dtype=int)
final_dataset = pd.concat([final_dataset, employment_dummies], axis=1)

final_scale_cols = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "repayment_history",
    "transaction_count",
    "spending_ratio",
    "regional_risk_index",
    "local_unemployment_rate",
    "join_year",
    "join_month",
    "join_day",
    "education_level_encoded",
    "gender_encoded",
    "join_weekday_encoded",
    "income_group_encoded",
    "repayment_history_quantile_bin",
    "transaction_kmeans_bin",
    "debt_to_income_ratio",
    "average_monthly_transactions",
    "spending_to_income_ratio",
    "spending_log",
    "spending_sqrt",
    "spending_reciprocal",
    "annual_income_boxcox",
    "loan_amount_yeojohnson",
]

final_dataset[final_scale_cols] = RobustScaler().fit_transform(final_dataset[final_scale_cols])

final_dataset = final_dataset.drop(
    columns=[
        "gender",
        "region",
        "loan_purpose",
        "employment_type",
        "education_level",
        "join_date",
        "join_weekday",
        "income_group",
        "transaction_cluster_raw",
    ]
)

final_dataset.to_csv("outputs/final_cleaned_transformed_dataset.csv", index=False)
print("Final dataset saved in outputs/final_cleaned_transformed_dataset.csv")
print("Final shape:", final_dataset.shape)
print("Missing values left:", final_dataset.isna().sum().sum())
final_dataset.head()

Final dataset saved in outputs/final_cleaned_transformed_dataset.csv
Final shape: (60, 43)
Missing values left: 0


,customer_id,age,annual_income,loan_amount,credit_score,transaction_count,spending_ratio,default_flag,repayment_history,regional_risk_index,local_unemployment_rate,annual_income_missing_flag,credit_score_missing_flag,join_year,join_month,join_day,education_level_encoded,gender_encoded,region_East,region_North,region_South,region_West,loan_purpose_Business,loan_purpose_Car,loan_purpose_Education,loan_purpose_Home,loan_purpose_Other,credit_score_flag,repayment_history_quantile_bin,transaction_kmeans_bin,debt_to_income_ratio,average_monthly_transactions,spending_to_income_ratio,spending_log,spending_reciprocal,spending_sqrt,annual_income_boxcox,loan_amount_yeojohnson,join_weekday_encoded,income_group_encoded,employment_type_Salaried,employment_type_Self-Employed,employment_type_Unemployed
0,CUST001,-1.060000,-0.390621,-0.376337,-0.930035,-0.237726,1.173670,0,-0.666667,-0.923077,-0.923077,0,0,0.500000,-0.64,0.935484,0.0,-1.0,0,0,0,1,0,0,0,1,0,0,-0.5,0.0,-0.288359,-0.237726,1.173670,0.993516,-0.840745,1.077828,-0.501420,-0.462389,-0.75,-0.5,0,0,1
1,CUST002,0.673333,-0.352994,0.095942,0.228975,-0.816537,-0.208228,0,-0.333333,-0.538462,-0.461538,0,0,-0.833333,0.32,0.354839,0.0,1.0,0,1,0,0,1,0,0,0,0,1,0.0,-0.8,0.474536,-0.816537,-0.208228,-0.250363,0.294117,-0.229525,-0.444099,0.090812,0.00,-0.5,0,0,1
2,CUST003,0.340000,-0.799435,-0.728824,-0.435336,1.105943,0.430719,0,-0.333333,0.461538,0.538462,0,0,0.833333,-0.32,0.032258,-2.0,2.0,0,0,1,0,0,0,0,0,1,0,0.0,0.8,-0.444523,1.105943,0.430719,0.429530,-0.419782,0.431138,-1.336842,-1.204950,-0.75,-0.5,1,0,0
3,CUST004,-0.193333,-0.074562,-0.465289,-0.336396,0.392765,0.744048,1,0.333333,0.000000,0.000000,0,0,0.166667,0.16,0.935484,0.0,0.0,1,0,0,0,1,0,0,0,0,0,0.5,0.0,-0.730195,0.392765,0.744048,0.688816,-0.629445,0.716345,-0.081771,-0.608917,0.75,-0.5,1,0,0
4,CUST005,-0.193333,-0.544358,-0.326387,0.073498,0.837209,0.484257,1,2.000000,0.461538,0.538462,0,0,-0.833333,-0.64,-0.032258,-2.0,0.0,0,0,1,0,0,0,1,0,0,0,1.0,0.8,0.038845,0.837209,0.484257,0.476515,-0.460008,0.481364,-0.763317,-0.388007,-0.25,-0.5,0,1,0


Interpretation: The final dataset is ready for machine learning.

## Report Summary

- Missing values were handled using different methods.
- Outliers were checked with Z-score, IQR, percentile capping, and winsorization.
- Encoding, scaling, and transformations were applied.
- New features were created from the original columns.
- Final dataset is saved in `outputs/final_cleaned_transformed_dataset.csv`.